In [ ]:
# Transformers + PEFT + Datasets + BitsAndBytes
!pip install torch --upgrade
!pip install transformers --upgrade
!pip install datasets --upgrade
!pip install peft --upgrade
!pip install bitsandbytes --upgrade

# Optional / Utility
!pip install accelerate --upgrade        # For distributed / GPU training
!pip install sentencepiece --upgrade     # For some tokenizers
!pip install tqdm --upgrade              # Progress bars
!pip install matplotlib --upgrade        # Optional: for plotting loss or token distributions

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
model_name = "Qwen/Qwen2.5-1.5B"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "v_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # ~1% trainable

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [ ]:
data_files = {
    "train": "/content/train.jsonl",
    "validation": "/content/val.jsonl"
}

dataset = load_dataset("json", data_files=data_files)

def format_instruction(example):
    """
    Convert JSONL {instruction, input, output} into a single text string
    suitable for causal LM training.
    """
    instruction = example.get("instruction", "")
    inp = example.get("input", "")
    output = example.get("output", "")

    if inp.strip():
        text = f"### Instruction:\n{instruction}\n\n### Input:\n{inp}\n\n### Response:\n{output}"
    else:
        text = f"### Instruction:\n{instruction}\n\n### Response:\n{output}"

    return {"text": text}

dataset = dataset.map(format_instruction)

def tokenize_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)


print(tokenized_dataset)

Map:   0%|          | 0/3481 [00:00<?, ? examples/s]

Map:   0%|          | 0/387 [00:00<?, ? examples/s]

Map:   0%|          | 0/3481 [00:00<?, ? examples/s]

Map:   0%|          | 0/387 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 3481
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 387
    })
})


In [ ]:
training_args = TrainingArguments(
    output_dir="./qlora-output",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer
)

trainer.train()

/tmp/ipython-input-2711599551.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,3.302100
20,1.989700
30,1.238100
40,1.044600
50,0.996400
60,0.880300
70,0.988600
80,0.904100
90,0.904600
100,1.062100


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=654, training_loss=0.9497897989523885, metrics={'train_runtime': 4058.7614, 'train_samples_per_second': 2.573, 'train_steps_per_second': 0.161, 'total_flos': 4.210680735203328e+16, 'train_loss': 0.9497897989523885, 'epoch': 3.0})

In [ ]:
model.save_pretrained("./adapters/adapter_model")
tokenizer.save_pretrained("./adapters/adapter_model")
print("Adapter weights saved!")

Adapter weights saved!


In [ ]:
!zip -r adapter_model.zip ./adapters/adapter_model

  adding: adapters/adapter_model/ (stored 0%)
  adding: adapters/adapter_model/adapter_config.json (deflated 57%)
  adding: adapters/adapter_model/merges.txt (deflated 57%)
  adding: adapters/adapter_model/vocab.json (deflated 61%)
  adding: adapters/adapter_model/README.md (deflated 65%)
  adding: adapters/adapter_model/chat_template.jinja (deflated 71%)
  adding: adapters/adapter_model/tokenizer.json (deflated 81%)
  adding: adapters/adapter_model/tokenizer_config.json (deflated 89%)
  adding: adapters/adapter_model/added_tokens.json (deflated 67%)
  adding: adapters/adapter_model/adapter_model.safetensors (deflated 8%)
  adding: adapters/adapter_model/special_tokens_map.json (deflated 62%)
